#0. 환경설정

In [1]:
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import font_manager

# ==================== 한글 폰트 강제 설정 ====================
def setup_korean_font_force():
    """한글 폰트 강제 설정 (파일 경로 직접 지정)"""
    
    font_paths = [
        "C:/Windows/Fonts/malgun.ttf",
        "C:/Windows/Fonts/NanumGothic.ttf",
        "C:/Windows/Fonts/gulim.ttc",
        "/System/Library/Fonts/AppleGothic.ttf",  # macOS
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",  # Linux
    ]
    
    font_path = None
    for path in font_paths:
        if os.path.exists(path):
            font_path = path
            break
    
    if font_path:
        # 폰트 파일 직접 등록
        font_prop = font_manager.FontProperties(fname=font_path)
        font_name = font_prop.get_name()
        
        # FontManager에 추가
        font_manager.fontManager.addfont(font_path)
        
        # 기본 폰트로 설정
        plt.rcParams['font.family'] = font_name
        plt.rcParams['axes.unicode_minus'] = False
        
        print(f"✅ 한글 폰트 강제 설정: {font_name} ({os.path.basename(font_path)})")
    else:
        print("⚠️ 한글 폰트 파일을 찾을 수 없습니다.")
        print("   영문 레이블을 사용합니다.")
        plt.rcParams['font.family'] = 'DejaVu Sans'
        plt.rcParams['axes.unicode_minus'] = False

# 한글 폰트 강제 설정 실행
setup_korean_font_force()


✅ 한글 폰트 강제 설정: Malgun Gothic (malgun.ttf)


In [2]:
# 음성 라이브러리 

In [3]:
# pip install librosa soundfile scipy

In [4]:
import librosa          # 음성 신호 처리
import soundfile as sf  # WAV 파일 읽기/쓰기
from scipy import signal  # 신호 처리 (필터링 등)


In [5]:
import os
import sys
import platform
import warnings

# [1] Protobuf 에러 및 로그 레벨 설정 (최상단 고정)
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import tensorflow as tf
from tensorflow.keras import mixed_precision
import torch
import numpy as np

# 음성 처리 라이브러리 추가
try:
    import librosa
    import soundfile as sf
    from scipy import signal
    AUDIO_LIBS_AVAILABLE = True
except ImportError as e:
    AUDIO_LIBS_AVAILABLE = False
    print(f"⚠️ 음성 처리 라이브러리 설치 필요: {e}")

# 0. 경고 메시지 억제
warnings.filterwarnings("ignore")

print("=" * 70)
print("     🎤 음성 분류(Audio CNN/RNN) 학습 최적화 환경 설정 (VRAM 4GB)")
print("=" * 70)

# 1. TensorFlow 음성 학습 최적화 (메모리 관리)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # 동적 할당: 모델 크기에 따라 필요한 만큼만 점유
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        
        # 4GB VRAM 고려: 음성 데이터 (스펙트로그램)를 위한 메모리 할당
        # 음성은 시계열 데이터이므로 이미지보다 약간 적은 메모리 사용
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=1792)]  # 1.75GB
        )
        
        # [음성 CNN 핵심] 혼합 정밀도: 학습 속도↑, VRAM 사용량↓
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        
        print(f"✅ TF GPU: 음성 CNN/RNN 가속 모드 활성화 (Limit: 1.75GB / Mixed Precision)")
    except Exception as e:
        print(f"ℹ️ GPU 설정 정보: {e}")
else:
    print("⚠️ GPU를 사용할 수 없습니다. CPU 모드로 실행됩니다.")

# 2. PyTorch 및 기타 디바이스 확인
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n[디바이스 가용 상태]")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Audio CNN/RNN Acceleration: {'AVAILABLE (GPU)' if gpus else 'CPU ONLY'}")
print(f"음성 데이터 정규화 정책: float16 (Mixed Precision)")

# 3. 음성 처리 라이브러리 확인
print(f"\n[음성 처리 라이브러리 상태]")
if AUDIO_LIBS_AVAILABLE:
    print(f"✅ librosa: {librosa.__version__}")
    print(f"✅ soundfile: 설치됨")
    print(f"✅ scipy: 설치됨")
else:
    print(f"❌ 음성 처리 라이브러리 미설치")
    print(f"\n설치 명령어:")
    print(f"   pip install librosa soundfile scipy")

# 4. 음성 데이터 규격 가이드
print("\n" + "=" * 70)
print(f"💡 음성 CNN/RNN 모델 구성 팁:")
print(f" - 원시 음성: (24000,) shape, 16kHz, 1.5초")
print(f" - MFCC 특징: (40, 47) shape")
print(f" - 멜 스펙트로그램: (128, 47) shape")
print(f" - Batch Size: 32 또는 64 권장 (4GB VRAM 고려)")
print(f" - 모델 타입: 1D CNN, LSTM, GRU, Conv1D+LSTM 하이브리드")
print("=" * 70)

# 5. 데이터 전처리 권장사항
print(f"\n💡 음성 데이터 전처리 권장:")
print(f" 1. 음성 정규화: [-1, 1] 범위로 스케일링")
print(f" 2. MFCC 추출: librosa.feature.mfcc()")
print(f" 3. 멜 스펙트로그램: librosa.feature.melspectrogram()")
print(f" 4. 데이터 증강: 피치 시프트, 타임 스트레치, 노이즈 추가")
print(f" 5. Zero Padding: 고정 길이 맞추기")
print("=" * 70 + "\n")

# 6. 필수 라이브러리 설치 확인 및 안내
if not AUDIO_LIBS_AVAILABLE:
    print("\n" + "!" * 70)
    print("⚠️  경고: 음성 데이터셋 생성을 위해 다음 라이브러리가 필요합니다:")
    print("!" * 70)
    print("\n다음 명령어로 설치하세요:")
    print("\n  pip install librosa==0.10.1")
    print("  pip install soundfile==0.12.1")
    print("  pip install scipy")
    print("\n또는 한 번에 설치:")
    print("  pip install librosa soundfile scipy")
    print("\n" + "!" * 70 + "\n")


d:\anaconda3\envs\tf_gpu310_fintech\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


     🎤 음성 분류(Audio CNN/RNN) 학습 최적화 환경 설정 (VRAM 4GB)
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce GTX 1650 Ti, compute capability 7.5
✅ TF GPU: 음성 CNN/RNN 가속 모드 활성화 (Limit: 1.75GB / Mixed Precision)

[디바이스 가용 상태]
TensorFlow Version: 2.10.1
Audio CNN/RNN Acceleration: AVAILABLE (GPU)
음성 데이터 정규화 정책: float16 (Mixed Precision)

[음성 처리 라이브러리 상태]
✅ librosa: 0.11.0
✅ soundfile: 설치됨
✅ scipy: 설치됨

💡 음성 CNN/RNN 모델 구성 팁:
 - 원시 음성: (24000,) shape, 16kHz, 1.5초
 - MFCC 특징: (40, 47) shape
 - 멜 스펙트로그램: (128, 47) shape
 - Batch Size: 32 또는 64 권장 (4GB VRAM 고려)
 - 모델 타입: 1D CNN, LSTM, GRU, Conv1D+LSTM 하이브리드

💡 음성 데이터 전처리 권장:
 1. 음성 정규화: [-1, 1] 범위로 스케일링
 2. MFCC 추출: librosa.feature.mfcc()
 3. 멜 스펙트로그램: librosa.feature.melspectrogram()
 4. 데이터 증강: 피치 시프트, 타임 스트레치, 노이즈 추가
 5. Zero Padding: 고정 길이 맞추기



#1-1 데이터 생성

In [6]:
"""
=====================================================================
  1. 합성 음성 데이터셋 생성기 (dataset_generator.py)
  - 특성 함수 기반 합성 음성 9,000개 생성
  - 메타데이터 저장 (나중에 실제 음성 변환에 사용)
=====================================================================
"""

import os
import json
import random
import numpy as np
from datetime import datetime
import librosa
import soundfile as sf
from scipy import signal
import warnings
warnings.filterwarnings('ignore')

# ==================== 설정 ====================
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
DATASET_NAME = "robot_voice_synthetic"
DATA_DIR = f"{DATASET_NAME}_{timestamp}"

CATEGORIES = {
    'go': '앞으로 가',
    'stop': '멈춰'
}

# 음성 설정
SAMPLE_RATE = 16000  # 16kHz (음성 인식 표준)
BASE_DURATION = 1.5  # 기본 길이 1.5초

# ==================== 음성 특성 함수 클래스 ====================
class VoiceCharacteristics:
    """음성 특성 함수 - 각 샘플마다 고유한 음성 DNA"""
    
    def __init__(self, speaker_id, emotion, seed):
        """화자별, 감정별 고유 특성 초기화"""
        rng = np.random.RandomState(seed)
        
        # 1. 음원 특성 (Source Characteristics)
        self.f0_base = rng.uniform(80, 280)  # 기본 주파수 (Hz)
        self.f0_range = rng.uniform(20, 80)  # 억양 변화 범위 (Hz)
        self.jitter = rng.uniform(0.001, 0.02)  # 주파수 미세변동 (%)
        self.shimmer = rng.uniform(0.01, 0.08)  # 진폭 미세변동 (%)
        self.vibrato_rate = rng.uniform(4, 7)  # 비브라토 속도 (Hz)
        self.vibrato_depth = rng.uniform(0, 0.03)  # 비브라토 깊이
        
        # 2. 포먼트 특성 (Filter Characteristics)
        self.formant_f1 = rng.uniform(500, 900)   # 1번 포먼트 (Hz)
        self.formant_f2 = rng.uniform(1200, 2500) # 2번 포먼트 (Hz)
        self.formant_f3 = rng.uniform(2500, 3500) # 3번 포먼트 (Hz)
        self.formant_bw = rng.uniform(50, 150)    # 포먼트 대역폭 (Hz)
        
        # 3. 음색 특성 (Timbre Characteristics)
        self.harmonic_weights = self._generate_harmonic_profile(rng)
        self.spectral_tilt = rng.uniform(-12, -3)  # dB/octave
        self.breathiness = rng.uniform(0.0, 0.3)   # 기식성 (0~1)
        self.roughness = rng.uniform(0.0, 0.2)     # 거친 정도
        
        # 4. 운율 특성 (Prosody Characteristics)
        self.intonation_type = rng.choice([
            'rising', 'falling', 'rising-falling', 'flat'
        ])
        self.stress_pattern = rng.uniform(0.8, 1.5)  # 강세 강도
        
        # 5. 감정 변조 적용
        self._apply_emotion_modulation(emotion, rng)
        
        # 6. 발화 스타일
        self.articulation = rng.uniform(0.7, 1.3)  # 조음 명확도
        self.speech_rhythm = rng.choice(['syllable-timed', 'stress-timed'])
        self.pause_tendency = rng.uniform(0.0, 0.2)  # 휴지 경향
    
    def _generate_harmonic_profile(self, rng):
        """화자별 고유 배음 구조 생성"""
        decay_rate = rng.uniform(0.1, 0.5)
        profile = np.exp(-decay_rate * np.arange(1, 11))
        profile *= (1 + rng.uniform(-0.3, 0.3, size=10))
        emphasized = rng.choice([2, 3, 4, 5])
        profile[emphasized-1] *= rng.uniform(1.2, 1.5)
        return profile / profile[0]
    
    def _apply_emotion_modulation(self, emotion, rng):
        """감정에 따른 특성 변조"""
        emotion_profiles = {
            'neutral': {'f0_mult': 1.0, 'f0_var': 1.0, 'intensity': 1.0},
            'urgent': {'f0_mult': 1.15, 'f0_var': 1.5, 'intensity': 1.3},
            'calm': {'f0_mult': 0.9, 'f0_var': 0.6, 'intensity': 0.8},
            'tired': {'f0_mult': 0.85, 'f0_var': 0.5, 'intensity': 0.7},
            'excited': {'f0_mult': 1.2, 'f0_var': 1.8, 'intensity': 1.4}
        }
        
        profile = emotion_profiles.get(emotion, emotion_profiles['neutral'])
        self.f0_base *= profile['f0_mult']
        self.f0_range *= profile['f0_var']
        self.intensity = profile['intensity']
        
        if emotion == 'urgent':
            self.vibrato_rate *= 1.3
            self.breathiness *= 0.5
        elif emotion == 'tired':
            self.roughness *= 1.5
            self.breathiness *= 1.3
    
    def get_f0_contour(self, duration, text_type):
        """시간에 따른 F0 궤적 생성"""
        t = np.linspace(0, 1, int(duration * SAMPLE_RATE))
        
        if text_type == '앞으로 가':
            base_contour = 0.3 * t
        else:  # '멈춰'
            base_contour = -0.5 * t
        
        if self.intonation_type == 'rising':
            intonation = t
        elif self.intonation_type == 'falling':
            intonation = (1 - t)
        elif self.intonation_type == 'rising-falling':
            intonation = np.sin(np.pi * t)
        else:
            intonation = 0.1 * np.sin(2 * np.pi * t)
        
        contour = self.f0_base + self.f0_range * (base_contour + 0.5 * intonation)
        
        vibrato = self.vibrato_depth * self.f0_base * \
                  np.sin(2 * np.pi * self.vibrato_rate * t * duration)
        contour += vibrato
        
        jitter_noise = np.random.randn(len(t)) * self.jitter * self.f0_base
        contour += jitter_noise
        
        return np.maximum(contour, 50)
    
    def to_dict(self):
        """특성을 딕셔너리로 변환 (저장용)"""
        return {
            'f0_base': float(self.f0_base),
            'f0_range': float(self.f0_range),
            'jitter': float(self.jitter),
            'shimmer': float(self.shimmer),
            'vibrato_rate': float(self.vibrato_rate),
            'vibrato_depth': float(self.vibrato_depth),
            'formant_f1': float(self.formant_f1),
            'formant_f2': float(self.formant_f2),
            'formant_f3': float(self.formant_f3),
            'formant_bw': float(self.formant_bw),
            'harmonic_weights': self.harmonic_weights.tolist(),
            'spectral_tilt': float(self.spectral_tilt),
            'breathiness': float(self.breathiness),
            'roughness': float(self.roughness),
            'intonation_type': self.intonation_type,
            'stress_pattern': float(self.stress_pattern),
            'articulation': float(self.articulation),
            'speech_rhythm': self.speech_rhythm,
            'pause_tendency': float(self.pause_tendency),
            'intensity': float(self.intensity)
        }

# ==================== TTS 음성 생성 파라미터 ====================
class VoiceParameters:
    """음성 생성 파라미터"""
    
    SPEAKER_TYPES = [
        {'gender': 'male', 'age': 'young', 'pitch_shift': 0},
        {'gender': 'male', 'age': 'adult', 'pitch_shift': -2},
        {'gender': 'male', 'age': 'old', 'pitch_shift': -4},
        {'gender': 'female', 'age': 'young', 'pitch_shift': 4},
        {'gender': 'female', 'age': 'adult', 'pitch_shift': 3},
        {'gender': 'female', 'age': 'old', 'pitch_shift': 2},
        {'gender': 'child', 'age': 'child', 'pitch_shift': 6},
    ]
    
    SPEECH_RATES = {
        'very_slow': 0.7,
        'slow': 0.85,
        'normal': 1.0,
        'fast': 1.15,
        'very_fast': 1.3
    }
    
    EMOTIONS = ['neutral', 'urgent', 'calm', 'tired', 'excited']

# ==================== 음성 오염 레벨 정의 ====================
class VoiceContamination:
    """음성 오염 시뮬레이션 - 6단계"""
    
    LEVELS = {
        0: {'name': 'pristine', 'noise_db': 0, 'distortion': 0.0, 
            'reverb': 0.0, 'clarity': 1.0, 'description': '완전히 깨끗한 음성'},
        1: {'name': 'slight', 'noise_db': -30, 'distortion': 0.1, 
            'reverb': 0.15, 'clarity': 0.9, 'description': '약간의 배경 소음'},
        2: {'name': 'moderate', 'noise_db': -24, 'distortion': 0.2, 
            'reverb': 0.3, 'clarity': 0.75, 'description': '중간 정도의 소음'},
        3: {'name': 'heavy', 'noise_db': -18, 'distortion': 0.35, 
            'reverb': 0.45, 'clarity': 0.6, 'description': '심한 소음 환경'},
        4: {'name': 'severe', 'noise_db': -12, 'distortion': 0.5, 
            'reverb': 0.6, 'clarity': 0.45, 'description': '매우 심한 소음'},
        5: {'name': 'extreme', 'noise_db': -6, 'distortion': 0.7, 
            'reverb': 0.8, 'clarity': 0.3, 'description': '극단적 소음 환경'}
    }
    
    @staticmethod
    def sample_level(voice_type='normal'):
        """오염 레벨 샘플링 - 고오염(3-5) 가중치 4배"""
        if voice_type == 'emotional':
            weights = [0.2, 0.3, 0.5, 5.0, 6.0, 8.0]
        else:
            weights = [0.3, 0.5, 0.8, 4.0, 4.0, 4.0]
        return random.choices(range(6), weights=weights, k=1)[0]
    
    @staticmethod
    def get_params(level):
        """레벨에 따른 오염 파라미터 반환"""
        config = VoiceContamination.LEVELS[level]
        return {
            'level': level,
            'name': config['name'],
            'description': config['description'],
            'noise_db': config['noise_db'] + random.uniform(-2, 2),
            'distortion': config['distortion'] * random.uniform(0.8, 1.2),
            'reverb': config['reverb'] * random.uniform(0.8, 1.2),
            'clarity': config['clarity']
        }

# ==================== 음성 합성 함수 ====================
def apply_formant_filters(audio, characteristics, sr):
    """포먼트 필터 적용"""
    filtered = np.zeros_like(audio)
    formants = [
        (characteristics.formant_f1, 1.0),
        (characteristics.formant_f2, 0.6),
        (characteristics.formant_f3, 0.3)
    ]
    
    for freq, gain in formants:
        bw = characteristics.formant_bw
        low = max(50, freq - bw)
        high = min(sr/2 - 100, freq + bw)
        try:
            sos = signal.butter(2, [low, high], btype='band', fs=sr, output='sos')
            formant_signal = signal.sosfilt(sos, audio)
            filtered += formant_signal * gain
        except:
            filtered += audio * gain * 0.3
    return filtered

def apply_spectral_tilt(audio, tilt_db_per_octave, sr):
    """스펙트럼 기울기 적용"""
    fft = np.fft.rfft(audio)
    freqs = np.fft.rfftfreq(len(audio), 1/sr)
    tilt_db = tilt_db_per_octave * np.log2(np.maximum(freqs, 1) / 100)
    tilt_linear = 10 ** (tilt_db / 20)
    fft *= tilt_linear
    return np.fft.irfft(fft, n=len(audio))

def synthesize_base_audio_with_characteristics(text, characteristics, speech_rate):
    """특성 함수 기반 음성 합성"""
    duration = BASE_DURATION / speech_rate
    t = np.linspace(0, duration, int(SAMPLE_RATE * duration))
    
    # F0 궤적 생성
    f0_contour = characteristics.get_f0_contour(duration, text)
    
    # 음절 패턴
    if text == '앞으로 가':
        syllable_times = [0.0, 0.35, 0.7]
        syllable_durations = [0.25, 0.25, 0.3]
    else:
        syllable_times = [0.0, 0.4]
        syllable_durations = [0.3, 0.35]
    
    syllable_pattern = np.zeros_like(t)
    for start_time, dur in zip(syllable_times, syllable_durations):
        start_idx = int(start_time * SAMPLE_RATE)
        end_idx = start_idx + int(dur * SAMPLE_RATE)
        if end_idx < len(syllable_pattern):
            syllable_pattern[start_idx:end_idx] = 1.0
    
    # 배음 구조로 음원 생성
    audio = np.zeros_like(t)
    for harmonic_idx, weight in enumerate(characteristics.harmonic_weights):
        harmonic = harmonic_idx + 1
        phase = np.random.uniform(0, 2 * np.pi)
        audio += weight * np.sin(2 * np.pi * f0_contour * harmonic * t + phase)
    audio *= syllable_pattern
    
    # 기식성 추가
    if characteristics.breathiness > 0:
        breath_noise = np.random.randn(len(audio)) * characteristics.breathiness
        sos = signal.butter(2, 2000, btype='high', fs=SAMPLE_RATE, output='sos')
        breath_noise = signal.sosfilt(sos, breath_noise)
        audio += breath_noise * syllable_pattern * 0.3
    
    # 거친 정도 추가
    if characteristics.roughness > 0:
        rough_freq = np.random.uniform(50, 100)
        rough_mod = 1 + characteristics.roughness * 0.3 * np.sin(2 * np.pi * rough_freq * t)
        audio *= rough_mod
    
    # 포먼트 필터
    audio = apply_formant_filters(audio, characteristics, SAMPLE_RATE)
    
    # 스펙트럼 틸트
    audio = apply_spectral_tilt(audio, characteristics.spectral_tilt, SAMPLE_RATE)
    
    # 쉬머 (진폭 변동)
    shimmer_noise = 1 + np.random.randn(len(audio)) * characteristics.shimmer
    shimmer_noise = np.maximum(shimmer_noise, 0.5)
    audio *= shimmer_noise
    
    # 엔벨로프
    envelope = np.ones_like(audio)
    fade_samples = int(0.05 * SAMPLE_RATE)
    envelope[:fade_samples] = np.linspace(0, 1, fade_samples)
    envelope[-fade_samples:] = np.linspace(1, 0, fade_samples)
    audio *= envelope
    
    # 강도 적용
    audio *= characteristics.intensity
    
    # 정규화
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.8
    
    return audio

def extreme_audio_randomization(audio, params, characteristics):
    """극한 음성 도메인 랜더마이제이션"""
    audio = audio.copy()
    
    if random.random() > 0.3:
        pitch_shift = random.uniform(-1, 1)
        audio = librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=pitch_shift)
    
    if random.random() > 0.3:
        time_stretch = random.uniform(0.95, 1.05)
        audio = librosa.effects.time_stretch(audio, rate=time_stretch)
    
    if random.random() > 0.2:
        volume_factor = random.uniform(0.7, 1.2)
        audio = audio * volume_factor
    
    if random.random() > 0.4:
        filter_type = random.choice(['lowpass', 'highpass', 'bandpass'])
        if filter_type == 'lowpass':
            cutoff = random.uniform(2500, 4000)
            sos = signal.butter(3, cutoff, btype='low', fs=SAMPLE_RATE, output='sos')
        elif filter_type == 'highpass':
            cutoff = random.uniform(100, 300)
            sos = signal.butter(3, cutoff, btype='high', fs=SAMPLE_RATE, output='sos')
        else:
            low = random.uniform(200, 400)
            high = random.uniform(3000, 4000)
            sos = signal.butter(3, [low, high], btype='band', fs=SAMPLE_RATE, output='sos')
        audio = signal.sosfilt(sos, audio)
    
    if random.random() > 0.5:
        threshold = random.uniform(0.4, 0.7)
        audio = np.where(np.abs(audio) > threshold, threshold * np.sign(audio), audio)
    
    if random.random() > 0.7:
        clip_level = random.uniform(0.85, 0.95)
        audio = np.clip(audio, -clip_level, clip_level)
    
    return audio

def apply_voice_contamination(audio, params):
    """환경 소음 및 왜곡 적용"""
    contaminated = audio.copy()
    level = params['level']
    
    if level == 0:
        return contaminated
    
    noise_db = params['noise_db']
    distortion = params['distortion']
    reverb_amount = params['reverb']
    
    # 백색 잡음
    noise = np.random.randn(len(contaminated))
    noise_power = np.mean(audio ** 2) / (10 ** (noise_db / 10))
    noise = noise * np.sqrt(noise_power)
    contaminated += noise * (1 - params['clarity'])
    
    # 환경 소음
    if level >= 2:
        rumble_freq = random.uniform(30, 80)
        t = np.linspace(0, len(contaminated) / SAMPLE_RATE, len(contaminated))
        rumble = np.sin(2 * np.pi * rumble_freq * t) * 0.1 * distortion
        contaminated += rumble
    
    if level >= 3:
        hiss = np.random.randn(len(contaminated))
        sos = signal.butter(4, 4000, btype='high', fs=SAMPLE_RATE, output='sos')
        hiss = signal.sosfilt(sos, hiss)
        contaminated += hiss * 0.05 * distortion
    
    if level >= 4:
        num_clicks = random.randint(1, 5)
        for _ in range(num_clicks):
            click_pos = random.randint(0, len(contaminated) - 1)
            click_width = random.randint(1, 3)
            contaminated[click_pos:click_pos + click_width] += random.uniform(-0.5, 0.5)
    
    # 리버브
    if reverb_amount > 0:
        reverb_time = reverb_amount * 0.3
        reverb_samples = int(reverb_time * SAMPLE_RATE)
        if reverb_samples > 0:
            reverb_ir = np.exp(-np.linspace(0, 5, reverb_samples))
            contaminated = np.convolve(contaminated, reverb_ir, mode='same')
    
    # 왜곡
    if distortion > 0.3:
        contaminated = np.tanh(contaminated * (1 + distortion * 2))
    
    # 대역폭 제한
    if level >= 3:
        low_cut = random.randint(300, 500)
        high_cut = random.randint(2000, 3000)
        sos = signal.butter(3, [low_cut, high_cut], btype='band', 
                          fs=SAMPLE_RATE, output='sos')
        contaminated = signal.sosfilt(sos, contaminated)
    
    # 진폭 변조
    if level >= 4 and random.random() > 0.5:
        mod_freq = random.uniform(0.5, 3)
        t = np.linspace(0, len(contaminated) / SAMPLE_RATE, len(contaminated))
        modulation = 0.7 + 0.3 * np.sin(2 * np.pi * mod_freq * t)
        contaminated *= modulation
    
    # 정규화
    max_val = np.max(np.abs(contaminated))
    if max_val > 0:
        contaminated = contaminated / max_val * 0.9
    
    return contaminated

# ==================== 음성 샘플 생성 ====================
def generate_voice_sample(category, text, voice_type, base_seed):
    """음성 샘플 생성"""
    random.seed(base_seed)
    np.random.seed(base_seed)
    
    # 파라미터 선택
    speaker = random.choice(VoiceParameters.SPEAKER_TYPES)
    speaker_id = f"{speaker['gender']}_{speaker['age']}"
    
    speech_rate_name = random.choice(list(VoiceParameters.SPEECH_RATES.keys()))
    speech_rate = VoiceParameters.SPEECH_RATES[speech_rate_name]
    
    emotion = random.choice(VoiceParameters.EMOTIONS)
    
    level = VoiceContamination.sample_level(voice_type)
    contamination_params = VoiceContamination.get_params(level)
    
    # 특성 생성
    char_seed = base_seed + hash(speaker_id) % 10000
    characteristics = VoiceCharacteristics(speaker_id, emotion, char_seed)
    
    # 음성 합성
    audio = synthesize_base_audio_with_characteristics(text, characteristics, speech_rate)
    audio = extreme_audio_randomization(audio, contamination_params, characteristics)
    audio = apply_voice_contamination(audio, contamination_params)
    
    # 길이 조정
    target_length = int(BASE_DURATION * SAMPLE_RATE)
    if len(audio) > target_length:
        audio = audio[:target_length]
    else:
        audio = np.pad(audio, (0, target_length - len(audio)), mode='constant')
    
    metadata = {
        'category': category,
        'label': 0 if category == 'go' else 1,
        'text': text,
        'voice_type': voice_type,
        'speaker': speaker,
        'speech_rate': speech_rate_name,
        'emotion': emotion,
        'contamination': contamination_params,
        'characteristics': characteristics.to_dict(),
        'duration': BASE_DURATION,
        'sample_rate': SAMPLE_RATE,
        'source': 'synthetic'
    }
    
    return audio, metadata

# ==================== 데이터셋 생성 ====================
def generate_voice_dataset():
    """합성 음성 데이터셋 생성"""
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"\n📁 데이터셋 폴더 생성: {DATA_DIR}")
    
    for category in CATEGORIES:
        category_dir = os.path.join(DATA_DIR, category)
        os.makedirs(category_dir, exist_ok=True)
        print(f"   └─ {category}/ 폴더 생성")
    
    all_metadata = []
    SAMPLES_PER_TYPE = 2250
    
    for category, text in CATEGORIES.items():
        print(f"\n📁 카테고리: {category.upper()} ('{text}')")
        
        # 일반 음성
        print(f"   🎤 일반 음성 생성 중 ({SAMPLES_PER_TYPE}개)...")
        for idx in range(SAMPLES_PER_TYPE):
            seed = hash(f"{category}_normal_{idx}") % (2**32)
            
            try:
                audio, metadata = generate_voice_sample(category, text, 'normal', seed)
                
                file_name = f"audio_{category}_{idx:04d}_normal.wav"
                file_path = os.path.join(DATA_DIR, category, file_name)
                sf.write(file_path, audio, SAMPLE_RATE)
                
                metadata['file_name'] = file_name
                all_metadata.append(metadata)
                
                if (idx + 1) % 250 == 0:
                    print(f"      ✓ {idx + 1}/{SAMPLES_PER_TYPE} 완료")
            except Exception as e:
                print(f"      ⚠️ 생성 실패 ({idx}): {e}")
        
        # 감정 음성
        print(f"   🎭 감정 음성 생성 중 ({SAMPLES_PER_TYPE}개)...")
        for idx in range(SAMPLES_PER_TYPE):
            seed = hash(f"{category}_emotional_{idx}") % (2**32)
            
            try:
                audio, metadata = generate_voice_sample(category, text, 'emotional', seed)
                
                file_name = f"audio_{category}_{idx:04d}_emotional.wav"
                file_path = os.path.join(DATA_DIR, category, file_name)
                sf.write(file_path, audio, SAMPLE_RATE)
                
                metadata['file_name'] = file_name
                all_metadata.append(metadata)
                
                if (idx + 1) % 250 == 0:
                    print(f"      ✓ {idx + 1}/{SAMPLES_PER_TYPE} 완료")
            except Exception as e:
                print(f"      ⚠️ 생성 실패 ({idx}): {e}")
    
    # 메타데이터 저장
    metadata_path = os.path.join(DATA_DIR, 'dataset_metadata.json')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(all_metadata, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print(f"✅ 합성 음성 데이터셋 생성 완료!")
    print(f"{'='*60}")
    print(f"📂 경로: {os.path.abspath(DATA_DIR)}")
    print(f"📄 메타데이터: {metadata_path}")
    print(f"📊 총 샘플 수: {len(all_metadata)}개")
    
    return DATA_DIR, all_metadata

# ==================== 실행 ====================
if __name__ == "__main__":
    print(f"\n{'='*60}")
    print(f"🤖 1단계: 합성 음성 데이터셋 생성")
    print(f"{'='*60}")
    print(f"\n📋 설정:")
    print(f"   • 카테고리: {list(CATEGORIES.keys())}")
    print(f"   • 카테고리당 샘플: 4500개 (일반 2250 + 감정 2250)")
    print(f"   • 총 샘플 수: 9000개")
    print(f"   • 오염 레벨: 6단계 (고오염 70-80%)")
    
    dataset_dir, metadata = generate_voice_dataset()
    
    print(f"\n✅ 다음 단계:")
    print(f"   2단계: voice_converter.py 실행")
    print(f"   3단계: visualizer.py 실행")



🤖 1단계: 합성 음성 데이터셋 생성

📋 설정:
   • 카테고리: ['go', 'stop']
   • 카테고리당 샘플: 4500개 (일반 2250 + 감정 2250)
   • 총 샘플 수: 9000개
   • 오염 레벨: 6단계 (고오염 70-80%)

📁 데이터셋 폴더 생성: robot_voice_synthetic_20260221_233926
   └─ go/ 폴더 생성
   └─ stop/ 폴더 생성

📁 카테고리: GO ('앞으로 가')
   🎤 일반 음성 생성 중 (2250개)...
      ✓ 250/2250 완료
      ✓ 500/2250 완료
      ✓ 750/2250 완료
      ✓ 1000/2250 완료
      ✓ 1250/2250 완료
      ✓ 1500/2250 완료
      ✓ 1750/2250 완료
      ✓ 2000/2250 완료
      ✓ 2250/2250 완료
   🎭 감정 음성 생성 중 (2250개)...
      ✓ 250/2250 완료
      ✓ 500/2250 완료
      ✓ 750/2250 완료
      ✓ 1000/2250 완료
      ✓ 1250/2250 완료
      ✓ 1500/2250 완료
      ✓ 1750/2250 완료
      ✓ 2000/2250 완료
      ✓ 2250/2250 완료

📁 카테고리: STOP ('멈춰')
   🎤 일반 음성 생성 중 (2250개)...
      ✓ 250/2250 완료
      ✓ 500/2250 완료
      ✓ 750/2250 완료
      ✓ 1000/2250 완료
      ✓ 1250/2250 완료
      ✓ 1500/2250 완료
      ✓ 1750/2250 완료
      ✓ 2000/2250 완료
      ✓ 2250/2250 완료
   🎭 감정 음성 생성 중 (2250개)...
      ✓ 250/2250 완료
      ✓ 500/2250 완료
      ✓ 750/2250 완료


#1-2 데이터 생성

In [ ]:
"""
=====================================================================
  2. 실제 음성 변환기 (voice_converter.py)
  - 1단계에서 생성된 합성 데이터셋 로드
  - GUI로 실제 음성 파일 업로드 (WAV, MP3, FLAC 등)
  - 합성 음성의 특성을 실제 음성에 적용
  - 변환된 데이터셋 저장
=====================================================================
"""

import os
import json
import numpy as np
from datetime import datetime
import librosa
import soundfile as sf
from scipy import signal
import warnings
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
warnings.filterwarnings('ignore')

# ==================== 설정 ====================
SAMPLE_RATE = 16000
BASE_DURATION = 1.5

# ==================== 음성 특성 클래스 (복원용) ====================
class VoiceCharacteristics:
    """음성 특성 복원 클래스"""
    
    @classmethod
    def from_dict(cls, data):
        """딕셔너리에서 특성 복원"""
        obj = cls.__new__(cls)
        for key, value in data.items():
            if key == 'harmonic_weights':
                setattr(obj, key, np.array(value))
            else:
                setattr(obj, key, value)
        return obj

# ==================== 음성 변환 함수 ====================
def apply_formant_filters(audio, characteristics, sr):
    """포먼트 필터 적용"""
    filtered = np.zeros_like(audio)
    formants = [
        (characteristics.formant_f1, 1.0),
        (characteristics.formant_f2, 0.6),
        (characteristics.formant_f3, 0.3)
    ]
    
    for freq, gain in formants:
        bw = characteristics.formant_bw
        low = max(50, freq - bw)
        high = min(sr/2 - 100, freq + bw)
        try:
            sos = signal.butter(2, [low, high], btype='band', fs=sr, output='sos')
            formant_signal = signal.sosfilt(sos, audio)
            filtered += formant_signal * gain
        except:
            filtered += audio * gain * 0.3
    return filtered

def apply_spectral_tilt(audio, tilt_db_per_octave, sr):
    """스펙트럼 기울기 적용"""
    fft = np.fft.rfft(audio)
    freqs = np.fft.rfftfreq(len(audio), 1/sr)
    tilt_db = tilt_db_per_octave * np.log2(np.maximum(freqs, 1) / 100)
    tilt_linear = 10 ** (tilt_db / 20)
    fft *= tilt_linear
    return np.fft.irfft(fft, n=len(audio))

def extreme_audio_randomization(audio, params, characteristics):
    """극한 음성 도메인 랜더마이제이션"""
    audio = audio.copy()
    
    if np.random.random() > 0.3:
        pitch_shift = np.random.uniform(-1, 1)
        audio = librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=pitch_shift)
    
    if np.random.random() > 0.3:
        time_stretch = np.random.uniform(0.95, 1.05)
        audio = librosa.effects.time_stretch(audio, rate=time_stretch)
    
    if np.random.random() > 0.2:
        volume_factor = np.random.uniform(0.7, 1.2)
        audio = audio * volume_factor
    
    if np.random.random() > 0.4:
        filter_type = np.random.choice(['lowpass', 'highpass', 'bandpass'])
        if filter_type == 'lowpass':
            cutoff = np.random.uniform(2500, 4000)
            sos = signal.butter(3, cutoff, btype='low', fs=SAMPLE_RATE, output='sos')
        elif filter_type == 'highpass':
            cutoff = np.random.uniform(100, 300)
            sos = signal.butter(3, cutoff, btype='high', fs=SAMPLE_RATE, output='sos')
        else:
            low = np.random.uniform(200, 400)
            high = np.random.uniform(3000, 4000)
            sos = signal.butter(3, [low, high], btype='band', fs=SAMPLE_RATE, output='sos')
        audio = signal.sosfilt(sos, audio)
    
    if np.random.random() > 0.5:
        threshold = np.random.uniform(0.4, 0.7)
        audio = np.where(np.abs(audio) > threshold, threshold * np.sign(audio), audio)
    
    if np.random.random() > 0.7:
        clip_level = np.random.uniform(0.85, 0.95)
        audio = np.clip(audio, -clip_level, clip_level)
    
    return audio

def apply_voice_contamination(audio, params):
    """환경 소음 및 왜곡 적용"""
    contaminated = audio.copy()
    level = params['level']
    
    if level == 0:
        return contaminated
    
    noise_db = params['noise_db']
    distortion = params['distortion']
    reverb_amount = params['reverb']
    
    # 백색 잡음
    noise = np.random.randn(len(contaminated))
    noise_power = np.mean(audio ** 2) / (10 ** (noise_db / 10))
    noise = noise * np.sqrt(noise_power)
    contaminated += noise * (1 - params['clarity'])
    
    # 환경 소음
    if level >= 2:
        rumble_freq = np.random.uniform(30, 80)
        t = np.linspace(0, len(contaminated) / SAMPLE_RATE, len(contaminated))
        rumble = np.sin(2 * np.pi * rumble_freq * t) * 0.1 * distortion
        contaminated += rumble
    
    if level >= 3:
        hiss = np.random.randn(len(contaminated))
        sos = signal.butter(4, 4000, btype='high', fs=SAMPLE_RATE, output='sos')
        hiss = signal.sosfilt(sos, hiss)
        contaminated += hiss * 0.05 * distortion
    
    if level >= 4:
        num_clicks = np.random.randint(1, 5)
        for _ in range(num_clicks):
            click_pos = np.random.randint(0, len(contaminated) - 1)
            click_width = np.random.randint(1, 3)
            contaminated[click_pos:click_pos + click_width] += np.random.uniform(-0.5, 0.5)
    
    # 리버브
    if reverb_amount > 0:
        reverb_time = reverb_amount * 0.3
        reverb_samples = int(reverb_time * SAMPLE_RATE)
        if reverb_samples > 0:
            reverb_ir = np.exp(-np.linspace(0, 5, reverb_samples))
            contaminated = np.convolve(contaminated, reverb_ir, mode='same')
    
    # 왜곡
    if distortion > 0.3:
        contaminated = np.tanh(contaminated * (1 + distortion * 2))
    
    # 대역폭 제한
    if level >= 3:
        low_cut = np.random.randint(300, 500)
        high_cut = np.random.randint(2000, 3000)
        sos = signal.butter(3, [low_cut, high_cut], btype='band', 
                          fs=SAMPLE_RATE, output='sos')
        contaminated = signal.sosfilt(sos, contaminated)
    
    # 진폭 변조
    if level >= 4 and np.random.random() > 0.5:
        mod_freq = np.random.uniform(0.5, 3)
        t = np.linspace(0, len(contaminated) / SAMPLE_RATE, len(contaminated))
        modulation = 0.7 + 0.3 * np.sin(2 * np.pi * mod_freq * t)
        contaminated *= modulation
    
    # 정규화
    max_val = np.max(np.abs(contaminated))
    if max_val > 0:
        contaminated = contaminated / max_val * 0.9
    
    return contaminated

def apply_characteristics_to_real_voice(real_audio, real_sr, characteristics, 
                                       contamination_params, speech_rate, seed):
    """
    실제 사람 음성에 합성 음성의 특성 적용
    
    Args:
        real_audio: 실제 음성 샘플
        real_sr: 실제 음성의 샘플레이트
        characteristics: VoiceCharacteristics 객체
        contamination_params: 오염 파라미터
        speech_rate: 발화 속도
        seed: 랜덤 시드
    """
    np.random.seed(seed)
    
    # 1. 샘플레이트 변환
    if real_sr != SAMPLE_RATE:
        audio = librosa.resample(real_audio, orig_sr=real_sr, target_sr=SAMPLE_RATE)
    else:
        audio = real_audio.copy()
    
    # 2. 길이 조정 (속도 변환)
    target_length = int(BASE_DURATION * SAMPLE_RATE)
    current_length = len(audio)
    
    if current_length > 0:
        length_ratio = target_length / current_length
        audio = librosa.effects.time_stretch(audio, rate=1/length_ratio)
    
    # 3. 피치 변환
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            audio, 
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=SAMPLE_RATE
        )
        real_f0_mean = np.nanmean(f0[voiced_flag])
        
        if not np.isnan(real_f0_mean) and real_f0_mean > 0:
            f0_shift_semitones = 12 * np.log2(characteristics.f0_base / real_f0_mean)
            audio = librosa.effects.pitch_shift(
                audio, 
                sr=SAMPLE_RATE, 
                n_steps=f0_shift_semitones
            )
    except:
        # F0 추정 실패 시 기본 피치 변환
        pitch_shift = (characteristics.f0_base - 150) / 10
        audio = librosa.effects.pitch_shift(audio, sr=SAMPLE_RATE, n_steps=pitch_shift)
    
    # 4. 포먼트 특성 적용
    audio = apply_formant_filters(audio, characteristics, SAMPLE_RATE)
    
    # 5. 스펙트럼 틸트 적용
    audio = apply_spectral_tilt(audio, characteristics.spectral_tilt, SAMPLE_RATE)
    
    # 6. 기식성 추가
    if characteristics.breathiness > 0:
        breath_noise = np.random.randn(len(audio)) * characteristics.breathiness
        sos = signal.butter(2, 2000, btype='high', fs=SAMPLE_RATE, output='sos')
        breath_noise = signal.sosfilt(sos, breath_noise)
        audio += breath_noise * 0.2
    
    # 7. 거친 정도 추가
    if characteristics.roughness > 0:
        rough_freq = np.random.uniform(50, 100)
        t = np.linspace(0, len(audio) / SAMPLE_RATE, len(audio))
        rough_mod = 1 + characteristics.roughness * 0.3 * np.sin(2 * np.pi * rough_freq * t)
        audio *= rough_mod
    
    # 8. 강도 조정
    audio *= characteristics.intensity
    
    # 9. 도메인 랜더마이제이션
    audio = extreme_audio_randomization(audio, contamination_params, characteristics)
    
    # 10. 오염 적용
    audio = apply_voice_contamination(audio, contamination_params)
    
    # 11. 최종 길이 조정
    if len(audio) > target_length:
        audio = audio[:target_length]
    else:
        audio = np.pad(audio, (0, target_length - len(audio)), mode='constant')
    
    # 12. 정규화
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.8
    
    return audio

# ==================== GUI 애플리케이션 ====================
class VoiceConverterGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("🎤 실제 음성 변환기 - 2단계")
        self.root.geometry("700x650")
        self.root.resizable(False, False)
        
        self.dataset_dir = None
        self.go_voice_path = None
        self.stop_voice_path = None
        self.metadata = None
        
        self.setup_ui()
    
    def setup_ui(self):
        """UI 구성"""
        # 타이틀
        title_frame = tk.Frame(self.root, bg="#2c3e50", height=80)
        title_frame.pack(fill=tk.X)
        
        title_label = tk.Label(
            title_frame, 
            text="🎤 실제 음성 변환기",
            font=("Arial", 24, "bold"),
            bg="#2c3e50",
            fg="white"
        )
        title_label.pack(pady=20)
        
        # 메인 프레임
        main_frame = tk.Frame(self.root, padx=30, pady=20)
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Step 1: 데이터셋 폴더 선택
        step1_frame = tk.LabelFrame(
            main_frame, 
            text="Step 1: 합성 데이터셋 폴더 선택",
            font=("Arial", 12, "bold"),
            padx=15,
            pady=15
        )
        step1_frame.pack(fill=tk.X, pady=10)
        
        self.dataset_label = tk.Label(
            step1_frame,
            text="선택된 폴더: 없음",
            font=("Arial", 10),
            fg="gray"
        )
        self.dataset_label.pack(anchor=tk.W, pady=5)
        
        dataset_btn = tk.Button(
            step1_frame,
            text="📁 데이터셋 폴더 선택",
            font=("Arial", 11),
            bg="#3498db",
            fg="white",
            padx=20,
            pady=10,
            command=self.select_dataset
        )
        dataset_btn.pack(pady=5)
        
        # Step 2: 실제 음성 파일 업로드
        step2_frame = tk.LabelFrame(
            main_frame,
            text="Step 2: 실제 음성 파일 업로드 (WAV, MP3, FLAC 지원)",
            font=("Arial", 12, "bold"),
            padx=15,
            pady=15
        )
        step2_frame.pack(fill=tk.X, pady=10)
        
        # GO 음성
        go_frame = tk.Frame(step2_frame)
        go_frame.pack(fill=tk.X, pady=5)
        
        self.go_label = tk.Label(
            go_frame,
            text="'앞으로 가' 음성: 없음",
            font=("Arial", 10),
            fg="gray",
            width=40,
            anchor=tk.W
        )
        self.go_label.pack(side=tk.LEFT, padx=5)
        
        go_btn = tk.Button(
            go_frame,
            text="🎵 GO 업로드",
            font=("Arial", 10),
            bg="#27ae60",
            fg="white",
            padx=15,
            pady=5,
            command=self.upload_go_voice
        )
        go_btn.pack(side=tk.RIGHT)
        
        # STOP 음성
        stop_frame = tk.Frame(step2_frame)
        stop_frame.pack(fill=tk.X, pady=5)
        
        self.stop_label = tk.Label(
            stop_frame,
            text="'멈춰' 음성: 없음",
            font=("Arial", 10),
            fg="gray",
            width=40,
            anchor=tk.W
        )
        self.stop_label.pack(side=tk.LEFT, padx=5)
        
        stop_btn = tk.Button(
            stop_frame,
            text="🎵 STOP 업로드",
            font=("Arial", 10),
            bg="#e74c3c",
            fg="white",
            padx=15,
            pady=5,
            command=self.upload_stop_voice
        )
        stop_btn.pack(side=tk.RIGHT)
        
        # Step 3: 변환 시작
        step3_frame = tk.LabelFrame(
            main_frame,
            text="Step 3: 변환 시작",
            font=("Arial", 12, "bold"),
            padx=15,
            pady=15
        )
        step3_frame.pack(fill=tk.X, pady=10)
        
        self.convert_btn = tk.Button(
            step3_frame,
            text="🚀 변환 시작",
            font=("Arial", 12, "bold"),
            bg="#f39c12",
            fg="white",
            padx=30,
            pady=15,
            command=self.start_conversion,
            state=tk.DISABLED
        )
        self.convert_btn.pack(pady=10)
        
        # 진행상황
        progress_frame = tk.Frame(main_frame)
        progress_frame.pack(fill=tk.X, pady=10)
        
        self.progress_label = tk.Label(
            progress_frame,
            text="대기 중...",
            font=("Arial", 10),
            fg="gray"
        )
        self.progress_label.pack(pady=5)
        
        self.progress_bar = ttk.Progressbar(
            progress_frame,
            mode='determinate',
            length=600
        )
        self.progress_bar.pack(pady=5)
        
        # 상태 텍스트
        self.status_text = tk.Text(
            main_frame,
            height=6,
            font=("Courier", 9),
            bg="#ecf0f1",
            state=tk.DISABLED
        )
        self.status_text.pack(fill=tk.BOTH, expand=True, pady=10)
    
    def log_status(self, message):
        """상태 로그 출력"""
        self.status_text.config(state=tk.NORMAL)
        self.status_text.insert(tk.END, message + "\n")
        self.status_text.see(tk.END)
        self.status_text.config(state=tk.DISABLED)
        self.root.update()
    
    def select_dataset(self):
        """데이터셋 폴더 선택"""
        folder = filedialog.askdirectory(title="합성 데이터셋 폴더 선택")
        if folder:
            # 메타데이터 파일 확인
            metadata_path = os.path.join(folder, 'dataset_metadata.json')
            if not os.path.exists(metadata_path):
                messagebox.showerror(
                    "오류",
                    "선택한 폴더에 dataset_metadata.json 파일이 없습니다.\n"
                    "1단계에서 생성된 데이터셋 폴더를 선택하세요."
                )
                return
            
            # 메타데이터 로드
            try:
                with open(metadata_path, 'r', encoding='utf-8') as f:
                    self.metadata = json.load(f)
                
                self.dataset_dir = folder
                self.dataset_label.config(
                    text=f"선택된 폴더: {os.path.basename(folder)}",
                    fg="green"
                )
                self.log_status(f"✅ 데이터셋 로드 완료: {len(self.metadata)}개 샘플")
                self.check_ready()
            except Exception as e:
                messagebox.showerror("오류", f"메타데이터 로드 실패:\n{e}")
    
    def upload_go_voice(self):
        """GO 음성 파일 업로드 (WAV, MP3, FLAC 지원)"""
        file_path = filedialog.askopenfilename(
            title="'앞으로 가' 음성 파일 선택",
            filetypes=[
                ("오디오 파일", "*.wav *.mp3 *.flac *.ogg *.m4a"),
                ("WAV 파일", "*.wav"),
                ("MP3 파일", "*.mp3"),
                ("FLAC 파일", "*.flac"),
                ("OGG 파일", "*.ogg"),
                ("모든 파일", "*.*")
            ]
        )
        if file_path:
            self.go_voice_path = file_path
            self.go_label.config(
                text=f"'앞으로 가' 음성: {os.path.basename(file_path)}",
                fg="green"
            )
            self.log_status(f"✅ GO 음성 업로드: {os.path.basename(file_path)}")
            self.check_ready()
    
    def upload_stop_voice(self):
        """STOP 음성 파일 업로드 (WAV, MP3, FLAC 지원)"""
        file_path = filedialog.askopenfilename(
            title="'멈춰' 음성 파일 선택",
            filetypes=[
                ("오디오 파일", "*.wav *.mp3 *.flac *.ogg *.m4a"),
                ("WAV 파일", "*.wav"),
                ("MP3 파일", "*.mp3"),
                ("FLAC 파일", "*.flac"),
                ("OGG 파일", "*.ogg"),
                ("모든 파일", "*.*")
            ]
        )
        if file_path:
            self.stop_voice_path = file_path
            self.stop_label.config(
                text=f"'멈춰' 음성: {os.path.basename(file_path)}",
                fg="green"
            )
            self.log_status(f"✅ STOP 음성 업로드: {os.path.basename(file_path)}")
            self.check_ready()
    
    def check_ready(self):
        """변환 준비 확인"""
        if self.dataset_dir and self.go_voice_path and self.stop_voice_path:
            self.convert_btn.config(state=tk.NORMAL)
            self.progress_label.config(text="준비 완료! 변환 시작 버튼을 클릭하세요.", fg="green")
    
    def start_conversion(self):
        """변환 시작 (스레드)"""
        self.convert_btn.config(state=tk.DISABLED)
        self.progress_label.config(text="변환 중...", fg="blue")
        
        # 백그라운드 스레드에서 실행
        thread = threading.Thread(target=self.convert_voices)
        thread.start()
    
    def convert_voices(self):
        """실제 음성 변환 실행"""
        try:
            # 출력 폴더 생성
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            output_dir = f"robot_voice_converted_{timestamp}"
            os.makedirs(output_dir, exist_ok=True)
            
            for category in ['go', 'stop']:
                category_dir = os.path.join(output_dir, category)
                os.makedirs(category_dir, exist_ok=True)
            
            self.log_status(f"\n📁 출력 폴더 생성: {output_dir}")
            
            # 실제 음성 로드
            voice_files = {
                'go': self.go_voice_path,
                'stop': self.stop_voice_path
            }
            
            real_voices = {}
            for category, path in voice_files.items():
                self.log_status(f"🎵 {category.upper()} 음성 로드 중...")
                audio, sr = librosa.load(path, sr=None)
                real_voices[category] = (audio, sr)
                self.log_status(f"   ✅ 샘플레이트: {sr}Hz, 길이: {len(audio)/sr:.2f}초")
            
            # 변환 실행
            converted_metadata = []
            total_samples = len(self.metadata)
            
            self.log_status(f"\n🔄 변환 시작 (총 {total_samples}개 샘플)...")
            
            for idx, meta in enumerate(self.metadata):
                category = meta['category']
                
                # 실제 음성 가져오기
                real_audio, real_sr = real_voices[category]
                
                # 특성 복원
                characteristics = VoiceCharacteristics.from_dict(meta['characteristics'])
                contamination_params = meta['contamination']
                
                # 속도
                speech_rates = {
                    'very_slow': 0.7, 'slow': 0.85, 'normal': 1.0,
                    'fast': 1.15, 'very_fast': 1.3
                }
                speech_rate = speech_rates.get(meta['speech_rate'], 1.0)
                
                # 시드 (재현성)
                seed = hash(meta['file_name']) % (2**32)
                
                # 변환 실행
                converted_audio = apply_characteristics_to_real_voice(
                    real_audio, real_sr, characteristics,
                    contamination_params, speech_rate, seed
                )
                
                # 저장
                output_path = os.path.join(output_dir, category, meta['file_name'])
                sf.write(output_path, converted_audio, SAMPLE_RATE)
                
                # 메타데이터 업데이트
                meta['source'] = 'real_voice_converted'
                meta['original_voice_file'] = os.path.basename(voice_files[category])
                converted_metadata.append(meta)
                
                # 진행상황 업데이트
                progress = (idx + 1) / total_samples * 100
                self.progress_bar['value'] = progress
                
                if (idx + 1) % 500 == 0:
                    self.log_status(f"   진행: {idx + 1}/{total_samples} ({progress:.1f}%)")
                
                self.root.update()
            
            # 메타데이터 저장
            metadata_path = os.path.join(output_dir, 'converted_metadata.json')
            with open(metadata_path, 'w', encoding='utf-8') as f:
                json.dump(converted_metadata, f, ensure_ascii=False, indent=2)
            
            self.log_status(f"\n{'='*50}")
            self.log_status(f"✅ 변환 완료!")
            self.log_status(f"📂 출력 경로: {os.path.abspath(output_dir)}")
            self.log_status(f"📊 총 샘플 수: {len(converted_metadata)}개")
            self.log_status(f"{'='*50}")
            
            self.progress_label.config(text="변환 완료!", fg="green")
            
            messagebox.showinfo(
                "완료",
                f"변환이 완료되었습니다!\n\n"
                f"출력 폴더: {output_dir}\n"
                f"총 샘플: {len(converted_metadata)}개\n\n"
                f"다음 단계: visualizer.py를 실행하여 결과를 확인하세요."
            )
            
        except Exception as e:
            self.log_status(f"\n❌ 오류 발생: {e}")
            import traceback
            self.log_status(traceback.format_exc())
            messagebox.showerror("오류", f"변환 중 오류가 발생했습니다:\n{e}")
        
        finally:
            self.convert_btn.config(state=tk.NORMAL)

# ==================== 실행 ====================
if __name__ == "__main__":
    print(f"\n{'='*60}")
    print(f"🤖 2단계: 실제 음성 변환기")
    print(f"{'='*60}")
    print(f"\n📋 지원 포맷:")
    print(f"   • WAV (무손실)")
    print(f"   • MP3 (손실 압축)")
    print(f"   • FLAC (무손실 압축)")
    print(f"   • OGG, M4A")
    print(f"\n💡 사용 방법:")
    print(f"   1. 1단계에서 생성된 데이터셋 폴더 선택")
    print(f"   2. '앞으로 가' 음성 파일 업로드 (go.wav/mp3)")
    print(f"   3. '멈춰' 음성 파일 업로드 (stop.wav/mp3)")
    print(f"   4. 변환 시작 버튼 클릭")
    print(f"\n📌 GUI 창이 열립니다...")
    
    root = tk.Tk()
    app = VoiceConverterGUI(root)
    root.mainloop()



🤖 2단계: 실제 음성 변환기

📋 지원 포맷:
   • WAV (무손실)
   • MP3 (손실 압축)
   • FLAC (무손실 압축)
   • OGG, M4A

💡 사용 방법:
   1. 1단계에서 생성된 데이터셋 폴더 선택
   2. '앞으로 가' 음성 파일 업로드 (go.wav/mp3)
   3. '멈춰' 음성 파일 업로드 (stop.wav/mp3)
   4. 변환 시작 버튼 클릭

📌 GUI 창이 열립니다...


In [ ]:
"""
=====================================================================
  3. 데이터셋 시각화 도구 (visualizer.py)
  - 1단계 합성 데이터셋 분석 및 시각화
  - 2단계 변환 데이터셋 분석 및 시각화
  - 비교 차트 및 스펙트로그램 생성
  - 진행상황 실시간 표시
=====================================================================
"""

import os
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # GUI 없이 실행
import matplotlib.pyplot as plt
import librosa
import librosa.display
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (matplotlib)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False

# ==================== 설정 ====================
SAMPLE_RATE = 16000

# ==================== 메타데이터 로드 ====================
def load_metadata(dataset_dir):
    """데이터셋 메타데이터 로드"""
    print(f"\n🔍 메타데이터 검색 중...")
    
    # 합성 데이터셋
    synthetic_meta_path = os.path.join(dataset_dir, 'dataset_metadata.json')
    # 변환 데이터셋
    converted_meta_path = os.path.join(dataset_dir, 'converted_metadata.json')
    
    metadata = {}
    
    if os.path.exists(synthetic_meta_path):
        print(f"   📄 합성 메타데이터 로드 중...")
        with open(synthetic_meta_path, 'r', encoding='utf-8') as f:
            metadata['synthetic'] = json.load(f)
        print(f"   ✅ 합성 데이터셋: {len(metadata['synthetic'])}개 샘플")
    
    if os.path.exists(converted_meta_path):
        print(f"   📄 변환 메타데이터 로드 중...")
        with open(converted_meta_path, 'r', encoding='utf-8') as f:
            metadata['converted'] = json.load(f)
        print(f"   ✅ 변환 데이터셋: {len(metadata['converted'])}개 샘플")
    
    if not metadata:
        raise FileNotFoundError(
            "메타데이터 파일을 찾을 수 없습니다.\n"
            "dataset_metadata.json 또는 converted_metadata.json 파일이 있는 폴더를 선택하세요."
        )
    
    return metadata

# ==================== 통계 분석 ====================
def analyze_dataset_statistics(metadata_list, dataset_name):
    """데이터셋 통계 분석"""
    print(f"\n{'='*60}")
    print(f"📊 {dataset_name} 통계 분석 중...")
    print(f"{'='*60}")
    
    total_count = len(metadata_list)
    print(f"   • 총 샘플 수: {total_count}개")
    
    print(f"   🔄 카테고리 분석 중...")
    # 카테고리별 분포
    category_counts = {}
    for meta in metadata_list:
        cat = meta['category']
        category_counts[cat] = category_counts.get(cat, 0) + 1
    
    print(f"   📁 카테고리별 분포:")
    for cat, count in sorted(category_counts.items()):
        percentage = (count / total_count) * 100
        print(f"      {cat:>4}: {count:>4}개 ({percentage:>5.1f}%)")
    
    print(f"   🔄 오염 레벨 분석 중...")
    # 오염 레벨 분포
    level_counts = {}
    for meta in metadata_list:
        level = meta['contamination']['level']
        level_counts[level] = level_counts.get(level, 0) + 1
    
    print(f"   📈 오염 레벨 분포:")
    level_names = {
        0: 'pristine', 1: 'slight', 2: 'moderate',
        3: 'heavy', 4: 'severe', 5: 'extreme'
    }
    for level in sorted(level_counts.keys()):
        count = level_counts[level]
        percentage = (count / total_count) * 100
        name = level_names.get(level, 'unknown')
        print(f"      레벨 {level} ({name:>8}): {count:>4}개 ({percentage:>5.1f}%)")
    
    print(f"   🔄 화자 분석 중...")
    # 화자 분포
    speaker_counts = {}
    for meta in metadata_list:
        speaker_key = f"{meta['speaker']['gender']}_{meta['speaker']['age']}"
        speaker_counts[speaker_key] = speaker_counts.get(speaker_key, 0) + 1
    
    print(f"   👥 화자 분포:")
    for speaker, count in sorted(speaker_counts.items()):
        percentage = (count / total_count) * 100
        print(f"      {speaker:>15}: {count:>4}개 ({percentage:>5.1f}%)")
    
    print(f"   🔄 감정 분석 중...")
    # 감정 분포
    emotion_counts = {}
    for meta in metadata_list:
        emotion = meta['emotion']
        emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1
    
    print(f"   😊 감정 분포:")
    for emotion, count in sorted(emotion_counts.items()):
        percentage = (count / total_count) * 100
        print(f"      {emotion:>8}: {count:>4}개 ({percentage:>5.1f}%)")
    
    print(f"   🔄 속도 분석 중...")
    # 속도 분포
    speed_counts = {}
    for meta in metadata_list:
        speed = meta['speech_rate']
        speed_counts[speed] = speed_counts.get(speed, 0) + 1
    
    print(f"   ⚡ 발화 속도 분포:")
    for speed, count in sorted(speed_counts.items()):
        percentage = (count / total_count) * 100
        print(f"      {speed:>10}: {count:>4}개 ({percentage:>5.1f}%)")
    
    print(f"   ✅ {dataset_name} 통계 분석 완료!")
    
    return {
        'total': total_count,
        'categories': category_counts,
        'levels': level_counts,
        'speakers': speaker_counts,
        'emotions': emotion_counts,
        'speeds': speed_counts
    }

# ==================== 시각화: 분포 차트 ====================
def plot_distribution_charts(stats_dict, output_dir):
    """분포 차트 생성"""
    print(f"\n📊 분포 차트 생성 중...")
    
    num_datasets = len(stats_dict)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Dataset Distribution Analysis', fontsize=16, fontweight='bold')
    
    colors = ['#3498db', '#e74c3c']
    dataset_names = list(stats_dict.keys())
    
    print(f"   🔄 1/6 카테고리 분포 차트...")
    # 1. 카테고리 분포
    ax = axes[0, 0]
    categories = ['go', 'stop']
    x = np.arange(len(categories))
    width = 0.35
    
    for idx, (name, stats) in enumerate(stats_dict.items()):
        counts = [stats['categories'].get(cat, 0) for cat in categories]
        offset = width * (idx - 0.5 * (num_datasets - 1))
        ax.bar(x + offset, counts, width, label=name, color=colors[idx], alpha=0.8)
    
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.set_title('Category Distribution')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    print(f"   🔄 2/6 오염 레벨 분포 차트...")
    # 2. 오염 레벨 분포
    ax = axes[0, 1]
    levels = list(range(6))
    x = np.arange(len(levels))
    
    for idx, (name, stats) in enumerate(stats_dict.items()):
        counts = [stats['levels'].get(level, 0) for level in levels]
        offset = width * (idx - 0.5 * (num_datasets - 1))
        ax.bar(x + offset, counts, width, label=name, color=colors[idx], alpha=0.8)
    
    ax.set_xlabel('Contamination Level')
    ax.set_ylabel('Count')
    ax.set_title('Contamination Level Distribution')
    ax.set_xticks(x)
    ax.set_xticklabels([f'L{i}' for i in levels])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    print(f"   🔄 3/6 화자 분포 차트...")
    # 3. 화자 분포
    ax = axes[0, 2]
    first_stats = list(stats_dict.values())[0]
    speakers = sorted(first_stats['speakers'].keys())
    x = np.arange(len(speakers))
    
    for idx, (name, stats) in enumerate(stats_dict.items()):
        counts = [stats['speakers'].get(sp, 0) for sp in speakers]
        offset = width * (idx - 0.5 * (num_datasets - 1))
        ax.bar(x + offset, counts, width, label=name, color=colors[idx], alpha=0.8)
    
    ax.set_xlabel('Speaker Type')
    ax.set_ylabel('Count')
    ax.set_title('Speaker Distribution')
    ax.set_xticks(x)
    ax.set_xticklabels(speakers, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    print(f"   🔄 4/6 감정 분포 차트...")
    # 4. 감정 분포
    ax = axes[1, 0]
    emotions = sorted(first_stats['emotions'].keys())
    x = np.arange(len(emotions))
    
    for idx, (name, stats) in enumerate(stats_dict.items()):
        counts = [stats['emotions'].get(em, 0) for em in emotions]
        offset = width * (idx - 0.5 * (num_datasets - 1))
        ax.bar(x + offset, counts, width, label=name, color=colors[idx], alpha=0.8)
    
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Count')
    ax.set_title('Emotion Distribution')
    ax.set_xticks(x)
    ax.set_xticklabels(emotions, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    print(f"   🔄 5/6 속도 분포 차트...")
    # 5. 속도 분포
    ax = axes[1, 1]
    speeds = sorted(first_stats['speeds'].keys())
    x = np.arange(len(speeds))
    
    for idx, (name, stats) in enumerate(stats_dict.items()):
        counts = [stats['speeds'].get(sp, 0) for sp in speeds]
        offset = width * (idx - 0.5 * (num_datasets - 1))
        ax.bar(x + offset, counts, width, label=name, color=colors[idx], alpha=0.8)
    
    ax.set_xlabel('Speech Rate')
    ax.set_ylabel('Count')
    ax.set_title('Speech Rate Distribution')
    ax.set_xticks(x)
    ax.set_xticklabels(speeds, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    print(f"   🔄 6/6 총 샘플 수 비교 차트...")
    # 6. 총 샘플 수 비교
    ax = axes[1, 2]
    names = list(stats_dict.keys())
    totals = [stats['total'] for stats in stats_dict.values()]
    bars = ax.bar(names, totals, color=colors[:len(names)], alpha=0.8)
    
    ax.set_ylabel('Total Samples')
    ax.set_title('Total Sample Count')
    ax.grid(axis='y', alpha=0.3)
    
    # 값 표시
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, 'distribution_charts.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"   ✅ 저장 완료: {output_path}")
    plt.close()

# ==================== 시각화: 스펙트로그램 샘플 ====================
def plot_spectrogram_samples(metadata_dict, dataset_dir, output_dir):
    """레벨별 스펙트로그램 샘플"""
    print(f"\n🎵 스펙트로그램 샘플 생성 중...")
    
    for dataset_name, metadata_list in metadata_dict.items():
        print(f"\n   📁 {dataset_name} 데이터셋 처리 중...")
        
        # 각 레벨별, 카테고리별 샘플 선택
        print(f"      🔍 샘플 선택 중...")
        samples = {}
        for level in range(6):
            samples[level] = {}
            for category in ['go', 'stop']:
                matching = [
                    m for m in metadata_list 
                    if m['contamination']['level'] == level and m['category'] == category
                ]
                if matching:
                    samples[level][category] = matching[0]
        
        print(f"      📊 스펙트로그램 생성 중 (총 12개)...")
        # 스펙트로그램 생성
        fig = plt.figure(figsize=(16, 10))
        fig.suptitle(f'{dataset_name.upper()} - Sample Spectrograms by Contamination Level', 
                     fontsize=16, fontweight='bold')
        
        total_plots = 0
        for level in range(6):
            if level not in samples:
                continue
            
            for idx, category in enumerate(['go', 'stop']):
                if category not in samples[level]:
                    continue
                
                sample = samples[level][category]
                
                # 오디오 파일 경로
                audio_path = os.path.join(dataset_dir, sample['category'], sample['file_name'])
                if not os.path.exists(audio_path):
                    continue
                
                try:
                    # 오디오 로드
                    audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
                    
                    # 서브플롯
                    subplot_idx = level * 2 + idx + 1
                    ax = fig.add_subplot(3, 4, subplot_idx)
                    
                    # 스펙트로그램
                    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
                    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax, cmap='viridis')
                    
                    # 제목
                    contam = sample['contamination']
                    title = f"L{level} {category.upper()}\n"
                    title += f"SNR: {contam['noise_db']:.0f}dB"
                    ax.set_title(title, fontsize=9)
                    ax.set_xlabel('Time (s)', fontsize=8)
                    ax.set_ylabel('Frequency (Hz)', fontsize=8)
                    
                    total_plots += 1
                    print(f"         ✓ {total_plots}/12 완료 (L{level} {category})")
                    
                except Exception as e:
                    print(f"         ⚠️ 실패: {sample['file_name']} - {e}")
        
        plt.tight_layout()
        
        output_path = os.path.join(output_dir, f'{dataset_name}_spectrograms.png')
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"      ✅ 저장 완료: {output_path}")
        plt.close()

# ==================== 시각화: 특성 분포 히트맵 ====================
def plot_characteristics_heatmap(metadata_list, dataset_name, output_dir):
    """음성 특성 분포 히트맵"""
    print(f"\n🔥 {dataset_name} 특성 히트맵 생성 중...")
    
    # 특성 추출
    print(f"   🔄 특성 데이터 추출 중...")
    char_keys = [
        'f0_base', 'f0_range', 'jitter', 'shimmer',
        'formant_f1', 'formant_f2', 'formant_f3',
        'spectral_tilt', 'breathiness', 'roughness'
    ]
    
    char_data = {key: [] for key in char_keys}
    
    total = len(metadata_list)
    for idx, meta in enumerate(metadata_list):
        chars = meta['characteristics']
        for key in char_keys:
            if key in chars:
                char_data[key].append(chars[key])
        
        if (idx + 1) % 1000 == 0:
            print(f"      진행: {idx + 1}/{total} ({(idx+1)/total*100:.1f}%)")
    
    print(f"   🔄 데이터 정규화 중...")
    # 정규화 (0-1 범위)
    normalized_data = []
    labels = []
    
    for key, values in char_data.items():
        if values:
            arr = np.array(values)
            normalized = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
            normalized_data.append(normalized)
            labels.append(key)
    
    if not normalized_data:
        print(f"   ⚠️ 특성 데이터 없음")
        return
    
    print(f"   📊 히스토그램 생성 중...")
    # 히스토그램 행렬 생성
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(f'{dataset_name.upper()} - Voice Characteristics Distribution', 
                 fontsize=16, fontweight='bold')
    
    for idx, (key, values) in enumerate(zip(labels, normalized_data)):
        ax = axes[idx // 5, idx % 5]
        ax.hist(values, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
        ax.set_title(key, fontsize=10, fontweight='bold')
        ax.set_xlabel('Normalized Value', fontsize=8)
        ax.set_ylabel('Count', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        
        print(f"      ✓ {idx + 1}/10 완료 ({key})")
    
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f'{dataset_name}_characteristics_distribution.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"   ✅ 저장 완료: {output_path}")
    plt.close()

# ==================== 메인 실행 ====================
def visualize_datasets(dataset_dir):
    """데이터셋 시각화 메인 함수"""
    print(f"\n{'='*60}")
    print(f"📊 데이터셋 시각화 시작")
    print(f"{'='*60}")
    
    # 출력 폴더 생성
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = f"visualization_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n📁 출력 폴더: {output_dir}")
    
    # 메타데이터 로드
    metadata_dict = load_metadata(dataset_dir)
    
    # 통계 분석
    print(f"\n{'='*60}")
    print(f"1단계: 통계 분석")
    print(f"{'='*60}")
    stats_dict = {}
    for name, metadata_list in metadata_dict.items():
        stats_dict[name] = analyze_dataset_statistics(metadata_list, name)
    
    # 시각화 생성
    print(f"\n{'='*60}")
    print(f"2단계: 분포 차트 생성")
    print(f"{'='*60}")
    if len(stats_dict) > 0:
        plot_distribution_charts(stats_dict, output_dir)
    
    print(f"\n{'='*60}")
    print(f"3단계: 특성 히트맵 생성")
    print(f"{'='*60}")
    for name, metadata_list in metadata_dict.items():
        plot_characteristics_heatmap(metadata_list, name, output_dir)
    
    print(f"\n{'='*60}")
    print(f"4단계: 스펙트로그램 생성")
    print(f"{'='*60}")
    plot_spectrogram_samples(metadata_dict, dataset_dir, output_dir)
    
    print(f"\n{'='*60}")
    print(f"✅ 시각화 완료!")
    print(f"{'='*60}")
    print(f"\n📂 출력 폴더: {os.path.abspath(output_dir)}")
    print(f"\n생성된 파일:")
    for file in sorted(os.listdir(output_dir)):
        print(f"   • {file}")

# ==================== 실행 ====================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog
    
    print(f"\n{'='*60}")
    print(f"📊 3단계: 데이터셋 시각화 도구")
    print(f"{'='*60}")
    print(f"\n사용 방법:")
    print(f"   1. 데이터셋 폴더 선택")
    print(f"   2. 자동으로 분석 및 시각화 생성")
    print(f"\n💡 선택 가능한 폴더:")
    print(f"   • 1단계 합성 데이터셋 (robot_voice_synthetic_*)")
    print(f"   • 2단계 변환 데이터셋 (robot_voice_converted_*)")
    print(f"   • 두 데이터셋이 있는 상위 폴더")
    
    # GUI로 폴더 선택
    root = tk.Tk()
    root.withdraw()
    
    dataset_dir = filedialog.askdirectory(title="데이터셋 폴더 선택")
    
    if dataset_dir:
        try:
            visualize_datasets(dataset_dir)
            
            print(f"\n{'='*60}")
            print(f"🎉 모든 작업 완료!")
            print(f"{'='*60}")
            
        except Exception as e:
            print(f"\n❌ 오류 발생: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"\n⚠️ 폴더가 선택되지 않았습니다.")
